# 🧪 Temp: Position Mode 단발 테스트

축을 준비 상태로 만들고, Position 모드 이동을 딱 한 번 실행해서 결과를 확인하기 위한 임시 노트북입니다.

*Prerequisite: `01_wmx_system_startup.ipynb`으로 엔진/EtherCAT이 이미 켜져 있어야 합니다.*

In [ ]:
import rclpy
from wmx_r2_message.srv import SetAxis
from wmx_r2_message.msg import AxisPose, AxisState
from wmx_utils import WmxClient

if not rclpy.ok():
    rclpy.init()

wmx = WmxClient(node_name='wmx_client_temp')
print(f"✅ WMX Client Ready. Target Axes: {wmx.axis_list}")

## Step 1: 알람 클리어 + 서보 On + 호밍 + Position 모드 설정

축을 이동 가능한 상태로 준비하는 과정을 한 셀에 모았습니다. 순서대로:
1. 알람 클리어
2. 서보 On
3. 호밍 (현재 위치를 0으로 설정)
4. Position 모드 설정 (0 = Position)

In [ ]:
axes = wmx.axis_list
zeros = [0] * len(axes)

# 1. 알람 클리어
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/clear_alarm', req)
print(f"1) 알람 클리어: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 2. 서보 On
req = SetAxis.Request()
req.index = axes
req.data = [1] * len(axes)
res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
print(f"2) 서보 On: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 3. 호밍
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/homing', req)
print(f"3) 호밍: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

# 4. Position 모드 설정 (0 = Position)
req = SetAxis.Request()
req.index = axes
req.data = zeros
res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
print(f"4) Position 모드 설정: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

print("\n🚀 준비 완료. 다음 셀에서 이동을 실행하세요.")

## Step 2: Position 모드 이동 — 딱 한 번 실행

`TARGET_POSITION`을 원하는 값(라디안)으로 바꾼 뒤 이 셀을 실행하세요. 이동 전/후 실제 위치(`actual_pos`)를 비교해서 얼마나 움직였는지 보여줍니다.

⚠️ 이 셀을 다시 실행하면 "이전 위치 → 새 target"으로 또 이동합니다. 절대 위치 기준이라는 점에 주의하세요.

In [ ]:
import time

TARGET_POSITION = 10.0    # 목표 위치 (라디안)
MOVE_VELOCITY   = 5.0     # 이동 속도
MOVE_ACC        = 10.0    # 가속도
MOVE_DEC        = 10.0    # 감속도
WAIT_SEC        = 3.0     # 이동 완료를 기다릴 시간(초)

# 현재 위치를 읽어오는 헬퍼
_latest_state = {'msg': None}

def _state_callback(msg):
    _latest_state['msg'] = msg

_state_sub = wmx.create_subscription(AxisState, '/wmx/axis/state', _state_callback, 10)

def get_current_position():
    _latest_state['msg'] = None
    for _ in range(20):
        rclpy.spin_once(wmx, timeout_sec=0.1)
        if _latest_state['msg'] is not None:
            return list(_latest_state['msg'].actual_pos)
    raise RuntimeError("위치 정보를 받지 못했습니다. /wmx/axis/state가 발행 중인지 확인하세요.")

pos_pub = wmx.create_publisher(AxisPose, '/wmx/axis/position', 1)

before = get_current_position()
print(f"이동 전 위치: {before}")

msg = AxisPose()
msg.index = wmx.axis_list
msg.target = [TARGET_POSITION] * len(wmx.axis_list)
msg.velocity = [MOVE_VELOCITY] * len(wmx.axis_list)
msg.acc = [MOVE_ACC] * len(wmx.axis_list)
msg.dec = [MOVE_DEC] * len(wmx.axis_list)

print(f"🎯 목표 위치 {TARGET_POSITION}로 이동 중 ...")
pos_pub.publish(msg)
time.sleep(WAIT_SEC)

after = get_current_position()
traveled = [a - b for a, b in zip(after, before)]
turns = [t / 6.283185 for t in traveled]

print(f"이동 후 위치: {after}")
print(f"실제 이동 거리(rad): {[f'{t:.3f}' for t in traveled]}")
print(f"실제 이동 거리(바퀴): {[f'{t:.3f}' for t in turns]}")

## Step 3: Velocity 모드 상대 이동

Position 모드(Step 2)는 "어디 어디 지점으로 가라"는 **절대 위치** 명령이었죠. Velocity 모드는 반대로 목표 지점이라는 개념 자체가 없어요 — "이 속도로 돌아라"만 있고, 얼마나 이동했는지는 **속도 × 시간**으로 결과가 정해지는 상대적인 이동이에요.

이 셀은:
1. Position → Velocity 모드로 전환
2. 이동 전 위치 기록
3. `RUN_VELOCITY` 속도로 `RUN_SEC`초 동안 회전
4. 정지(속도 0) 명령
5. 이동 후 위치 기록 → 실제로 얼마나 움직였는지 비교
6. 다음 테스트를 위해 다시 Position 모드로 복귀

⚠️ 재실행하면 그 시점 위치에서 또 같은 방향/시간만큼 이동합니다 (상대 이동이라 매번 결과가 누적돼요).

In [ ]:
import time

RUN_VELOCITY = 3.0    # 목표 속도
RUN_ACC      = 10.0   # 가속도
RUN_DEC      = 10.0   # 감속도
RUN_SEC      = 3.0    # 이 속도로 실행할 시간(초)

# 1. Velocity 모드로 전환 (1 = Velocity)
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [1] * len(wmx.axis_list)
res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
print(f"Mode → Velocity: {'✅ ' + res.message if res.success else '❌ ' + res.message}")

vel_pub = wmx.create_publisher(AxisVelocity, '/wmx/axis/velocity', 1)

# 2. 이동 전 위치
before = get_current_position()
print(f"이동 전 위치: {before}")

# 3. RUN_SEC초 동안 회전
msg = AxisVelocity()
msg.index = wmx.axis_list
msg.velocity = [RUN_VELOCITY] * len(wmx.axis_list)
msg.acc = [RUN_ACC] * len(wmx.axis_list)
msg.dec = [RUN_DEC] * len(wmx.axis_list)

print(f"🚀 속도 {RUN_VELOCITY}로 {RUN_SEC}초 동안 회전 ...")
vel_pub.publish(msg)
time.sleep(RUN_SEC)

# 4. 정지 (속도 0)
msg.velocity = [0.0] * len(wmx.axis_list)
vel_pub.publish(msg)
print("🛑 정지")

# 5. 이동 후 위치 및 비교
after = get_current_position()
traveled = [a - b for a, b in zip(after, before)]
turns = [t / 6.283185 for t in traveled]

print(f"이동 후 위치: {after}")
print(f"실제 이동 거리(rad): {[f'{t:.3f}' for t in traveled]}")
print(f"실제 이동 거리(바퀴): {[f'{t:.3f}' for t in turns]}")
print(f"참고: 속도 x 시간 = {RUN_VELOCITY * RUN_SEC:.2f} rad (가속/감속 구간 제외한 이론값)")

# 6. 다음 테스트를 위해 Position 모드로 복귀
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [0] * len(wmx.axis_list)
res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
print(f"Mode → Position (복귀): {'✅ ' + res.message if res.success else '❌ ' + res.message}")